In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import unquote
from selenium.webdriver.support.ui import Select
import pandas as pd
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import datetime 
from tqdm import tqdm
import warnings
from openpyxl import load_workbook
import os
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")
from IPython.display import clear_output 

In [14]:
df_i=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\101_Other_Support\New folder\Input\Spark Plug Service Interval.xlsx")


In [15]:
df_i=df_i[["Searchkey_New","Covered?","Year","VIO Information"]]
df_i=df_i[(df_i["Covered?"]=="NotCovered") & (df_i["VIO Information"]=="VIO")]
df_i.drop_duplicates(inplace=True)
df_i.reset_index(inplace=True)
df_i

,index,Searchkey_New,Covered?,Year,VIO Information
0,67173,1985 Ford Ranger XLS 2L L4 GAS -,NotCovered,1985,VIO
1,67250,1985 GMC C3500 Sierra Classic 4.8L L6 GAS -,NotCovered,1985,VIO
2,67253,1985 GMC C3500 Sierra Classic 5.7L V8 GAS -,NotCovered,1985,VIO
3,67303,1985 GMC Jimmy Base 5L V8 GAS LE9,NotCovered,1985,VIO
4,67327,1985 GMC K2500 Base 5.7L V8 GAS -,NotCovered,1985,VIO
5,67340,1985 GMC K3500 High Sierra 7.4L V8 GAS -,NotCovered,1985,VIO
6,67390,1985 Honda Civic CRX 1.5L L4 GAS -,NotCovered,1985,VIO
7,67408,1985 Isuzu I-Mark DLX 1.5L L4 GAS -,NotCovered,1985,VIO
8,67451,1985 Jeep Wagoneer Base 2.5L L4 GAS -,NotCovered,1985,VIO
9,67574,1985 Oldsmobile Calais Base 3L V6 GAS -,NotCovered,1985,VIO


In [5]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.maximize_window()
wait=WebDriverWait(driver, 10)
driver.get('https://dh.identifix.com/id/login')
sleep(1)


In [6]:
username_field = driver.find_element(By.ID,'UserName')  
username_field.send_keys('defours6')
sleep(1)

password_field = driver.find_element(By.ID,'Password')  
password_field.send_keys('spectrum')
sleep(1)

Login = driver.find_element(By.ID,'Login')  
Login.click()
sleep(1)

home=driver.current_url
driver.get(home)


In [16]:
cols=["Sl.No"]
df= pd.DataFrame(columns=cols)

In [ ]:
# 20,542 Number of rows planned 
# 19,757 Number of rows planned 
# 16,592 Number of rows planned
# 13,739 Number of rows planned

In [18]:
for i in tqdm(range(len(df_i))):    
    driver.get(home)
    print(i)
    sleep(2)
    try:
        typebox=wait.until(EC.presence_of_element_located((By.ID,"CreateVehicle_suggest_box")))
        sleep(4)
        typebox.send_keys(df_i['Searchkey_New'][i])
        print("Searching for: ", df_i['Searchkey_New'][i])
    except:
        continue
    o2=wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME,"ui-menu-item")))
    #print("number of vehicle: ", len(o2))
    if(len(o2))<=1:
        o=wait.until(EC.presence_of_element_located((By.CLASS_NAME,"ui-menu-item")))
        o.click()
    else:
        o2[0].click() 
    sleep(2)
    try:
        VID=driver.current_url.split('VID=')[1].split("&")[0]  
    except:
        df.loc[i,'Sl.No']=i
        df.loc[i,'GivenApplication']=df_i['Searchkey_New'][i]
        df.loc[i,'Link']="The vehicle you selected does not have any documents related to it."
        continue 
    ROID=driver.current_url.split('ROID=')[1].split("&")[0]
    df.loc[i,'Sl.No']=i
    df.loc[i,'GivenApplication']=df_i['Searchkey_New'][i]
    df.loc[i,'Selected Model']=driver.find_element(By.CLASS_NAME,'vehicle-info').text
    df.loc[i,'Link']=f'https://dh.identifix.com/MaintenanceSchedules/Index?ROID={ROID}&VID={VID}&LocationId=4'
    clear_output(wait=True)

100%|██████████| 24/24 [06:52<00:00, 17.21s/it]


In [19]:
df["Date"]=datetime.datetime.now().strftime("%d-%b-%Y")

In [20]:
df=df[['Sl.No', 'GivenApplication', 'Selected Model','Link', 'Date']]
df

,Sl.No,GivenApplication,Selected Model,Link,Date
0,0,1985 Ford Ranger XLS 2L L4 GAS -,"1985 Ford Ranger XLS 2.0L, L4, Gas, Asp N, VIN...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
1,1,1985 GMC C3500 Sierra Classic 4.8L L6 GAS -,"1985 GMC C3500 Sierra Classic 4.8L, L6, Gas, V...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
2,2,1985 GMC C3500 Sierra Classic 5.7L V8 GAS -,"1985 GMC C3500 Sierra Classic 5.7L, V8, Gas, V...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
3,3,1985 GMC Jimmy Base 5L V8 GAS LE9,"1985 GMC S15 Jimmy 2.5L, L4, VIN E, 8V, FI, US...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
4,4,1985 GMC K2500 Base 5.7L V8 GAS -,"1985 GMC K2500 5.7L, V8, Gas, VIN L, 16V, Carb...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
5,5,1985 GMC K3500 High Sierra 7.4L V8 GAS -,"1985 GMC K3500 High Sierra 7.4L, V8, Gas, VIN ...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
6,6,1985 Honda Civic CRX 1.5L L4 GAS -,"1985 Honda Civic CRX 1.5L, L4, 12V, Carb, USA/...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
7,7,1985 Isuzu I-Mark DLX 1.5L L4 GAS -,"1985 Isuzu I-Mark DLX 1.5L, L4, VIN K, USA/Canada",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
8,8,1985 Jeep Wagoneer Base 2.5L L4 GAS -,"1985 Jeep Wagoneer 2.5L, L4, Gas, Asp N, VIN U...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026
9,9,1985 Oldsmobile Calais Base 3L V6 GAS -,"1985 Oldsmobile Calais 3.0L, V6, VIN L, 12V, U...",https://dh.identifix.com/MaintenanceSchedules/...,26-Jan-2026


In [21]:
from sympy import false
OFolder=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\101_Other_Support\New folder"
output_path = os.path.join(OFolder, 'IdentifixServiceInterval.xlsx')
book = load_workbook(output_path)

with pd.ExcelWriter(output_path,engine='openpyxl', mode='a',if_sheet_exists='overlay') as writer:  
    startrow = book['Links'].max_row
    df.to_excel(writer,index=False, sheet_name='Links',header=false,startrow=startrow)